In [3]:
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass

import gc
import json
import os
import random
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# =========================
# Device
# =========================

print("PyTorch:", torch.__version__)
print("MPS built:", torch.backends.mps.is_built())
print("MPS available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    DEVICE_NAME = "mps"

    # 如果某个 op 暂时不支持 MPS，允许回退 CPU
    os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DEVICE_NAME = "cuda"

else:
    DEVICE = torch.device("cpu")
    DEVICE_NAME = "cpu"

print("Using device:", DEVICE_NAME)

# =========================
# Config
# =========================

PROJECT = Path.cwd().parent  # 项目根目录（notebooks/ 的上一级）

TARGET = "Will_Buy_EV"
ID_COL = "id"

SEED = 42

# 第一次先测速
N_FOLDS = 2
N_EPOCHS = 1

# 测速成功以后改成：
# N_FOLDS = 5
# N_EPOCHS = 3

BATCH_SIZE = 2048
INFER_BATCH_SIZE = 8192
NUM_WORKERS = 0

HIDDEN_DIM = 256
N_BLOCKS = 2

LR = 2e-3
WEIGHT_DECAY = 1e-5

LABEL_SMOOTHING_SCHEDULE = [
    0.025,
    0.008,
    0.0,
]

DROPOUT_SCHEDULE = [
    0.012,
    0.004,
    0.001,
]

OUT_DIR = (
    PROJECT
    / "outputs"
    / (
        "v9_2_realmlp_mps_"
        + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    )
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

print("PROJECT:", PROJECT)
print("OUTPUT :", OUT_DIR)
print("Folds  :", N_FOLDS)
print("Epochs :", N_EPOCHS)

PyTorch: 2.14.0
MPS built: True
MPS available: True
CUDA available: False
Using device: mps
PROJECT: /Users/yw/unsw/Kaggle
OUTPUT : /Users/yw/unsw/Kaggle/outputs/v9_2_realmlp_mps_20261001_025046_412310
Folds  : 2
Epochs : 1


In [4]:
# =========================
# Load data
# =========================

train_path = PROJECT / "data/train.csv"
test_path = PROJECT / "data/test.csv"
sample_path = PROJECT / "data/sample_submission.csv"

if not train_path.exists():
    raise FileNotFoundError(train_path)

if not test_path.exists():
    raise FileNotFoundError(test_path)

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

sample = (
    pd.read_csv(sample_path)
    if sample_path.exists()
    else None
)

if TARGET not in train.columns:
    raise ValueError(
        f"找不到 target: {TARGET}\n"
        f"train columns = {train.columns.tolist()}"
    )

y = train[TARGET].map(
    {
        "No": 0,
        "Yes": 1,
    }
)

if y.isna().any():
    raise ValueError(
        "Target 不是预期的 No / Yes"
    )

y = y.astype(np.int8)

print("train:", train.shape)
print("test :", test.shape)
print("positive rate:", y.mean())


# =========================
# Feature engineering
# =========================

BASE_FEATURES = [
    "Age",
    "Gender",
    "Annual_Income_USD",
    "City_Type",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Current_Car_Type",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Home_Charging_Possible",
    "Environmental_Concern_Level",
    "Subsidy_Available",
    "Range_Anxiety_Level",
]


def yes_no_numeric(series):
    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "yes": 1.0,
                "no": 0.0,
                "true": 1.0,
                "false": 0.0,
                "1": 1.0,
                "0": 0.0,
            }
        )
        .fillna(0.0)
        .astype(np.float32)
    )


def engineer_features(frame):

    missing = [
        col
        for col in BASE_FEATURES
        if col not in frame.columns
    ]

    if missing:
        raise ValueError(
            f"缺少特征: {missing}"
        )

    f = frame[BASE_FEATURES].copy()

    # -------------------------
    # Numeric features
    # -------------------------

    num = pd.DataFrame(
        index=f.index
    )

    numeric_raw = [
        "Age",
        "Annual_Income_USD",
        "Daily_Commute_km",
        "Number_of_Cars_Owned",
        "Charging_Stations_Near_Home",
        "Charging_Stations_Near_Work",
        "Environmental_Concern_Level",
    ]

    for col in numeric_raw:
        num[col] = pd.to_numeric(
            f[col],
            errors="coerce",
        ).astype(np.float32)

    income = pd.to_numeric(
        f["Annual_Income_USD"],
        errors="coerce",
    )

    commute = pd.to_numeric(
        f["Daily_Commute_km"],
        errors="coerce",
    )

    age = pd.to_numeric(
        f["Age"],
        errors="coerce",
    )

    cars = pd.to_numeric(
        f["Number_of_Cars_Owned"],
        errors="coerce",
    )

    home = pd.to_numeric(
        f["Charging_Stations_Near_Home"],
        errors="coerce",
    )

    work = pd.to_numeric(
        f["Charging_Stations_Near_Work"],
        errors="coerce",
    )

    concern = pd.to_numeric(
        f["Environmental_Concern_Level"],
        errors="coerce",
    )

    subsidy = yes_no_numeric(
        f["Subsidy_Available"]
    )

    range_ord = (
        f["Range_Anxiety_Level"]
        .astype(str)
        .map(
            {
                "Low": 0.0,
                "Medium": 1.0,
                "High": 2.0,
            }
        )
        .fillna(1.0)
        .astype(np.float32)
    )

    num["income_log"] = np.log1p(
        income.clip(lower=0)
    )

    num["commute_log"] = np.log1p(
        commute.clip(lower=0)
    )

    num["commute_sq"] = commute ** 2

    num["age_sq"] = age ** 2

    num["charging_total"] = (
        home + work
    )

    num["charging_gap"] = (
        home - work
    )

    num["income_per_car"] = (
        income / (cars + 1.0)
    )

    num["income_per_commute"] = (
        income / (commute + 1.0)
    )

    num["subsidy_bin"] = subsidy
    num["range_ord"] = range_ord

    num["subsidy_x_income"] = (
        subsidy * income
    )

    num["subsidy_x_concern"] = (
        subsidy * concern
    )

    num["concern_x_range"] = (
        concern * range_ord
    )

    # 少量 piecewise features
    for knot in [
        31004,
        42000,
        80000,
        120000,
        170537,
    ]:

        num[
            f"income_hinge_{knot}"
        ] = np.maximum(
            income - knot,
            0.0,
        )

    num["income_31_42k"] = (
        (income >= 31004)
        & (income <= 41970)
    ).astype(np.float32)

    num["income_ge_170537"] = (
        income >= 170537
    ).astype(np.float32)

    num["commute_ge_83"] = (
        commute >= 83
    ).astype(np.float32)

    # -------------------------
    # Categorical features
    # -------------------------

    cat = pd.DataFrame(
        index=f.index
    )

    raw_cat = [
        "Gender",
        "City_Type",
        "Current_Car_Type",
        "Home_Charging_Possible",
        "Subsidy_Available",
        "Range_Anxiety_Level",
    ]

    for col in raw_cat:

        cat[col] = (
            f[col]
            .astype(str)
            .fillna("__NA__")
        )

    # discrete numeric → category
    discrete_cols = [
        "Number_of_Cars_Owned",
        "Charging_Stations_Near_Home",
        "Charging_Stations_Near_Work",
        "Environmental_Concern_Level",
    ]

    for col in discrete_cols:

        values = (
            pd.to_numeric(
                f[col],
                errors="coerce",
            )
            .fillna(-999)
        )

        cat[
            f"{col}__cat"
        ] = (
            np.rint(values)
            .astype(np.int64)
            .astype(str)
        )

    income_int = (
        income
        .fillna(-999)
        .round()
        .astype(np.int64)
    )

    commute_fill = (
        commute
        .fillna(-999)
    )

    cat["income_last_2"] = (
        income_int % 100
    ).astype(str)

    cat["income_10k"] = (
        income_int // 10000
    ).astype(str)

    cat["commute_integer"] = (
        np.rint(commute_fill)
        .astype(np.int64)
        .astype(str)
    )

    cat["age_5yr"] = (
        (
            pd.to_numeric(
                f["Age"],
                errors="coerce",
            )
            .fillna(-999)
            // 5
        )
        .astype(np.int64)
        .astype(str)
    )

    cat["subsidy__range"] = (
        cat["Subsidy_Available"]
        + "__"
        + cat["Range_Anxiety_Level"]
    )

    cat["subsidy__concern"] = (
        cat["Subsidy_Available"]
        + "__"
        + cat[
            "Environmental_Concern_Level__cat"
        ]
    )

    return num, cat


train_num, train_cat = engineer_features(
    train
)

test_num, test_cat = engineer_features(
    test
)

print(
    "numeric features:",
    train_num.shape[1],
)

print(
    "categorical features:",
    train_cat.shape[1],
)


# =========================
# Fold-local preprocessing
# =========================

@dataclass
class FoldPreprocessor:

    medians: np.ndarray
    lower: np.ndarray
    upper: np.ndarray
    means: np.ndarray
    stds: np.ndarray

    cat_maps: list
    cat_cardinalities: list


def fit_preprocessor(
    numeric_df,
    categorical_df,
):

    x = numeric_df.to_numpy(
        dtype=np.float64
    )

    medians = np.nanmedian(
        x,
        axis=0,
    )

    x = np.where(
        np.isnan(x),
        medians,
        x,
    )

    lower = np.nanquantile(
        x,
        0.005,
        axis=0,
    )

    upper = np.nanquantile(
        x,
        0.995,
        axis=0,
    )

    x = np.clip(
        x,
        lower,
        upper,
    )

    means = x.mean(axis=0)

    stds = x.std(axis=0)

    stds = np.where(
        stds < 1e-6,
        1.0,
        stds,
    )

    cat_maps = []
    cat_cardinalities = []

    for col in categorical_df.columns:

        values = (
            categorical_df[col]
            .astype(str)
        )

        categories = pd.Index(
            pd.unique(values)
        )

        mapping = {
            value: index + 1
            for index, value
            in enumerate(categories)
        }

        cat_maps.append(
            mapping
        )

        # 0 reserved for unknown
        cat_cardinalities.append(
            len(mapping) + 1
        )

    return FoldPreprocessor(
        medians=medians.astype(np.float32),
        lower=lower.astype(np.float32),
        upper=upper.astype(np.float32),
        means=means.astype(np.float32),
        stds=stds.astype(np.float32),
        cat_maps=cat_maps,
        cat_cardinalities=cat_cardinalities,
    )


def transform_features(
    numeric_df,
    categorical_df,
    prep,
):

    x = numeric_df.to_numpy(
        dtype=np.float32
    )

    x = np.where(
        np.isnan(x),
        prep.medians,
        x,
    )

    x = np.clip(
        x,
        prep.lower,
        prep.upper,
    )

    x = (
        x - prep.means
    ) / prep.stds

    x = np.clip(
        x,
        -8.0,
        8.0,
    ).astype(np.float32)

    categorical_arrays = []

    for i, col in enumerate(
        categorical_df.columns
    ):

        values = (
            categorical_df[col]
            .astype(str)
            .map(prep.cat_maps[i])
            .fillna(0)
            .astype(np.int64)
            .to_numpy()
        )

        categorical_arrays.append(
            values
        )

    cat_matrix = np.column_stack(
        categorical_arrays
    ).astype(np.int64)

    return x, cat_matrix

train: (668665, 15)
test : (286571, 14)
positive rate: 0.17464500160768098
numeric features: 28
categorical features: 16


In [6]:
# =========================
# Dataset
# =========================

class TabDataset(Dataset):

    def __init__(
        self,
        x_num,
        x_cat,
        y=None,
    ):

        self.x_num = torch.from_numpy(
            x_num
        )

        self.x_cat = torch.from_numpy(
            x_cat
        )

        self.y = (
            None
            if y is None
            else torch.from_numpy(
                np.asarray(
                    y,
                    dtype=np.float32,
                )
            )
        )

    def __len__(self):
        return len(self.x_num)

    def __getitem__(self, idx):

        if self.y is None:
            return (
                self.x_num[idx],
                self.x_cat[idx],
            )

        return (
            self.x_num[idx],
            self.x_cat[idx],
            self.y[idx],
        )


# =========================
# Model
# =========================

def embedding_dim(
    cardinality,
):

    return int(
        min(
            24,
            max(
                4,
                round(
                    (cardinality ** 0.25)
                    * 3
                ),
            ),
        )
    )


class ResidualBlock(nn.Module):

    def __init__(
        self,
        dim,
        dropout,
    ):

        super().__init__()

        self.norm = nn.LayerNorm(
            dim
        )

        self.fc1 = nn.Linear(
            dim,
            dim * 2,
        )

        self.fc2 = nn.Linear(
            dim * 2,
            dim,
        )

        self.activation = nn.SiLU()

        self.dropout = nn.Dropout(
            dropout
        )

    def set_dropout(
        self,
        p,
    ):

        self.dropout.p = float(p)

    def forward(
        self,
        x,
    ):

        h = self.norm(x)

        h = self.fc1(h)

        h = self.activation(h)

        h = self.dropout(h)

        h = self.fc2(h)

        h = self.dropout(h)

        return x + h


class RealMLPMac(nn.Module):

    def __init__(
        self,
        n_numeric,
        cat_cardinalities,
        hidden_dim=256,
        n_blocks=2,
        dropout=0.01,
    ):

        super().__init__()

        self.embeddings = nn.ModuleList()

        embedding_dims = []

        for cardinality in cat_cardinalities:

            dim = embedding_dim(
                cardinality
            )

            self.embeddings.append(
                nn.Embedding(
                    cardinality,
                    dim,
                )
            )

            embedding_dims.append(
                dim
            )

        categorical_dim = sum(
            embedding_dims
        )

        self.numeric_norm = nn.LayerNorm(
            n_numeric
        )

        input_dim = (
            n_numeric
            + categorical_dim
        )

        self.input_layer = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim,
            ),

            nn.SiLU(),

            nn.Dropout(
                dropout
            ),
        )

        self.blocks = nn.ModuleList(
            [
                ResidualBlock(
                    hidden_dim,
                    dropout,
                )
                for _ in range(
                    n_blocks
                )
            ]
        )

        self.final_norm = nn.LayerNorm(
            hidden_dim
        )

        self.head = nn.Linear(
            hidden_dim,
            1,
        )

        # initialization
        for module in self.modules():

            if isinstance(
                module,
                nn.Linear,
            ):

                nn.init.xavier_uniform_(
                    module.weight
                )

                if module.bias is not None:

                    nn.init.zeros_(
                        module.bias
                    )

    def set_dropout(
        self,
        p,
    ):

        self.input_layer[-1].p = float(
            p
        )

        for block in self.blocks:

            block.set_dropout(
                p
            )

    def forward(
        self,
        x_num,
        x_cat,
    ):

        x_num = self.numeric_norm(
            x_num
        )

        categorical_parts = [
            embedding(
                x_cat[:, i]
            )
            for i, embedding
            in enumerate(
                self.embeddings
            )
        ]

        x = torch.cat(
            [
                x_num,
                *categorical_parts,
            ],
            dim=1,
        )

        x = self.input_layer(
            x
        )

        for block in self.blocks:

            x = block(
                x
            )

        x = self.final_norm(
            x
        )

        return self.head(
            x
        ).squeeze(1)


# =========================
# Training helpers
# =========================

def set_seed(
    seed,
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


@torch.no_grad()
def predict_loader(
    model,
    loader,
):

    model.eval()

    predictions = []

    for (
        x_num,
        x_cat,
    ) in loader:

        x_num = x_num.to(
            DEVICE
        )

        x_cat = x_cat.to(
            DEVICE
        )

        logits = model(
            x_num,
            x_cat,
        )

        probabilities = torch.sigmoid(
            logits
        )

        predictions.append(
            probabilities
            .cpu()
            .numpy()
        )

    return np.concatenate(
        predictions
    )


def binary_cross_entropy_smoothed(
    logits,
    targets,
    smoothing,
):

    if smoothing > 0:

        targets = (
            targets
            * (1.0 - smoothing)
            + 0.5 * smoothing
        )

    return (
        nn.functional
        .binary_cross_entropy_with_logits(
            logits,
            targets,
        )
    )


def train_one_fold(
    fold,
    train_indices,
    valid_indices,
):

    set_seed(
        SEED + fold
    )

    fold_start = time.time()

    # -------------------------
    # Fold preprocessing
    # -------------------------

    prep = fit_preprocessor(

        train_num.iloc[
            train_indices
        ],

        train_cat.iloc[
            train_indices
        ],
    )

    x_train_num, x_train_cat = (
        transform_features(

            train_num.iloc[
                train_indices
            ],

            train_cat.iloc[
                train_indices
            ],

            prep,
        )
    )

    x_valid_num, x_valid_cat = (
        transform_features(

            train_num.iloc[
                valid_indices
            ],

            train_cat.iloc[
                valid_indices
            ],

            prep,
        )
    )

    x_test_num, x_test_cat = (
        transform_features(
            test_num,
            test_cat,
            prep,
        )
    )

    y_train = (
        y.iloc[
            train_indices
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    y_valid = (
        y.iloc[
            valid_indices
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    # -------------------------
    # DataLoader
    # -------------------------

    train_loader = DataLoader(

        TabDataset(
            x_train_num,
            x_train_cat,
            y_train,
        ),

        batch_size=BATCH_SIZE,

        shuffle=True,

        num_workers=NUM_WORKERS,
    )

    valid_loader = DataLoader(

        TabDataset(
            x_valid_num,
            x_valid_cat,
        ),

        batch_size=INFER_BATCH_SIZE,

        shuffle=False,

        num_workers=NUM_WORKERS,
    )

    test_loader = DataLoader(

        TabDataset(
            x_test_num,
            x_test_cat,
        ),

        batch_size=INFER_BATCH_SIZE,

        shuffle=False,

        num_workers=NUM_WORKERS,
    )

    # -------------------------
    # Model
    # -------------------------

    model = RealMLPMac(

        n_numeric=
            x_train_num.shape[1],

        cat_cardinalities=
            prep.cat_cardinalities,

        hidden_dim=
            HIDDEN_DIM,

        n_blocks=
            N_BLOCKS,

        dropout=
            DROPOUT_SCHEDULE[0],

    ).to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=LR,

        weight_decay=
            WEIGHT_DECAY,
    )

    scheduler = (
        torch.optim
        .lr_scheduler
        .OneCycleLR(

            optimizer,

            max_lr=LR,

            total_steps=max(
                1,
                len(train_loader)
                * N_EPOCHS,
            ),

            pct_start=0.15,

            anneal_strategy="cos",
        )
    )

    best_auc = -np.inf
    best_epoch = None
    best_state = None

    epoch_records = []

    # -------------------------
    # Epoch loop
    # -------------------------

    for epoch in range(
        N_EPOCHS
    ):

        epoch_start = time.time()

        model.train()

        dropout = (
            DROPOUT_SCHEDULE[
                min(
                    epoch,
                    len(
                        DROPOUT_SCHEDULE
                    ) - 1,
                )
            ]
        )

        smoothing = (
            LABEL_SMOOTHING_SCHEDULE[
                min(
                    epoch,
                    len(
                        LABEL_SMOOTHING_SCHEDULE
                    ) - 1,
                )
            ]
        )

        model.set_dropout(
            dropout
        )

        losses = []

        for (
            x_num_batch,
            x_cat_batch,
            y_batch,
        ) in train_loader:

            x_num_batch = (
                x_num_batch.to(
                    DEVICE
                )
            )

            x_cat_batch = (
                x_cat_batch.to(
                    DEVICE
                )
            )

            y_batch = (
                y_batch.to(
                    DEVICE
                )
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                x_num_batch,
                x_cat_batch,
            )

            loss = (
                binary_cross_entropy_smoothed(
                    logits,
                    y_batch,
                    smoothing,
                )
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
            )

            optimizer.step()

            scheduler.step()

            losses.append(
                float(
                    loss
                    .detach()
                    .cpu()
                )
            )

        # -------------------------
        # Validation
        # -------------------------

        valid_prediction = (
            predict_loader(
                model,
                valid_loader,
            )
        )

        valid_auc = float(
            roc_auc_score(
                y_valid,
                valid_prediction,
            )
        )

        elapsed_minutes = (
            time.time()
            - epoch_start
        ) / 60.0

        improved = (
            valid_auc
            > best_auc
        )

        print(

            f"Fold {fold} | "
            f"epoch {epoch+1}/{N_EPOCHS} | "
            f"AUC={valid_auc:.6f} | "
            f"best={max(best_auc, valid_auc):.6f} | "
            f"loss={np.mean(losses):.5f} | "
            f"{elapsed_minutes:.1f} min"

            + (
                " ✓"
                if improved
                else ""
            )
        )

        epoch_records.append(
            {
                "fold": fold,
                "epoch": epoch + 1,
                "auc": valid_auc,
                "minutes":
                    elapsed_minutes,
            }
        )

        if improved:

            best_auc = (
                valid_auc
            )

            best_epoch = (
                epoch + 1
            )

            best_state = {
                key:
                    value
                    .detach()
                    .cpu()
                    .clone()

                for key, value
                in model
                .state_dict()
                .items()
            }

    # -------------------------
    # Restore best epoch
    # -------------------------

    model.load_state_dict(
        best_state
    )

    valid_prediction = (
        predict_loader(
            model,
            valid_loader,
        )
    )

    test_prediction = (
        predict_loader(
            model,
            test_loader,
        )
    )

    final_auc = float(
        roc_auc_score(
            y_valid,
            valid_prediction,
        )
    )

    total_minutes = (
        time.time()
        - fold_start
    ) / 60.0

    print(
        f"Fold {fold} DONE | "
        f"AUC={final_auc:.9f} | "
        f"best_epoch={best_epoch} | "
        f"total={total_minutes:.1f} min"
    )

    # -------------------------
    # Cleanup
    # -------------------------

    del (
        model,
        optimizer,
        scheduler,
        train_loader,
        valid_loader,
        test_loader,
    )

    gc.collect()

    if DEVICE_NAME == "mps":
        torch.mps.empty_cache()

    elif DEVICE_NAME == "cuda":
        torch.cuda.empty_cache()

    return (
        valid_prediction,
        test_prediction,
        final_auc,
        epoch_records,
    )

In [9]:
N_FOLDS = 5
N_EPOCHS = 3

In [10]:
# =========================
# Cross validation
# =========================

folds = list(

    StratifiedKFold(

        n_splits=
            N_FOLDS,

        shuffle=True,

        random_state=
            SEED,

    ).split(
        train,
        y,
    )
)

oof = np.full(
    len(train),
    np.nan,
    dtype=np.float64,
)

coverage = np.zeros(
    len(train),
    dtype=np.int8,
)

test_prediction_sum = (
    np.zeros(
        len(test),
        dtype=np.float64,
    )
)

fold_scores = []
epoch_scores = []

run_start = time.time()


for (
    fold,
    (
        train_indices,
        valid_indices,
    ),
) in enumerate(
    folds,
    start=1,
):

    print()
    print(
        "=" * 70
    )

    print(
        f"RealMLP Mac | "
        f"Fold {fold}/{N_FOLDS} | "
        f"device={DEVICE_NAME}"
    )

    print(
        "=" * 70
    )

    (
        valid_prediction,
        test_prediction,
        fold_auc,
        epoch_records,
    ) = train_one_fold(

        fold,

        train_indices,

        valid_indices,
    )

    oof[
        valid_indices
    ] = valid_prediction

    coverage[
        valid_indices
    ] += 1

    test_prediction_sum += (
        test_prediction
    )

    fold_scores.append(
        {
            "fold": fold,
            "auc": fold_auc,
        }
    )

    epoch_scores.extend(
        epoch_records
    )

    # -------------------------
    # Save partial result
    # -------------------------

    pd.DataFrame(
        fold_scores
    ).to_csv(
        OUT_DIR
        / "fold_scores_realmlp_mac.csv",
        index=False,
    )

    pd.DataFrame(
        epoch_scores
    ).to_csv(
        OUT_DIR
        / "epoch_scores_realmlp_mac.csv",
        index=False,
    )

    np.save(
        OUT_DIR
        / (
            f"oof_partial_"
            f"after_fold{fold}.npy"
        ),
        oof,
    )

    np.save(
        OUT_DIR
        / (
            f"test_sum_"
            f"after_fold{fold}.npy"
        ),
        test_prediction_sum,
    )


# =========================
# Validate
# =========================

if not np.all(
    coverage == 1
):

    raise ValueError(
        "OOF coverage incomplete"
    )


if not np.isfinite(
    oof
).all():

    raise ValueError(
        "OOF contains NaN"
    )


test_prediction = (
    test_prediction_sum
    / N_FOLDS
)


overall_auc = float(
    roc_auc_score(
        y,
        oof,
    )
)


total_runtime = (
    time.time()
    - run_start
) / 60.0


print()
print(
    "=" * 70
)

print(
    pd.DataFrame(
        fold_scores
    ).to_string(
        index=False,
        float_format=
            lambda value:
                f"{value:.9f}",
    )
)

print()

print(
    f"Overall RealMLP Mac OOF AUC "
    f"= {overall_auc:.9f}"
)

print(
    f"Total runtime "
    f"= {total_runtime:.1f} min"
)

print(
    "=" * 70
)


# =========================
# Save final files
# =========================

oof_frame = pd.DataFrame(
    {
        "id":
            train[
                "id"
            ].to_numpy(),

        "target":
            y.to_numpy(),

        "prediction":
            oof,
    }
)


submission = pd.DataFrame(
    {
        "id":
            test[
                "id"
            ].to_numpy(),

        TARGET:
            test_prediction,
    }
)


if sample is not None:

    submission = (
        sample[
            ["id"]
        ]
        .merge(

            submission,

            on="id",

            how="left",

            validate=
                "one_to_one",

            sort=False,
        )
    )


oof_path = (
    OUT_DIR
    / "oof_realmlp_mac.csv"
)

submission_path = (
    OUT_DIR
    / "submission_realmlp_mac.csv"
)


oof_frame.to_csv(
    oof_path,
    index=False,
)

submission.to_csv(
    submission_path,
    index=False,
)


report = {

    "device":
        DEVICE_NAME,

    "folds":
        N_FOLDS,

    "epochs":
        N_EPOCHS,

    "batch_size":
        BATCH_SIZE,

    "hidden_dim":
        HIDDEN_DIM,

    "n_blocks":
        N_BLOCKS,

    "overall_oof_auc":
        overall_auc,

    "total_runtime_minutes":
        total_runtime,

    "fold_scores":
        fold_scores,
}


report_path = (
    OUT_DIR
    / "realmlp_mac_report.json"
)


report_path.write_text(

    json.dumps(
        report,
        indent=2,
    ),

    encoding="utf-8",
)


print()
print(
    "Saved:"
)

print(
    oof_path
)

print(
    submission_path
)

print(
    report_path
)


RealMLP Mac | Fold 1/5 | device=mps
Fold 1 | epoch 1/3 | AUC=0.936827 | best=0.936827 | loss=0.28622 | 0.1 min ✓
Fold 1 | epoch 2/3 | AUC=0.937427 | best=0.937427 | loss=0.24494 | 0.1 min ✓
Fold 1 | epoch 3/3 | AUC=0.937799 | best=0.937799 | loss=0.22954 | 0.1 min ✓
Fold 1 DONE | AUC=0.937798619 | best_epoch=3 | total=0.4 min

RealMLP Mac | Fold 2/5 | device=mps
Fold 2 | epoch 1/3 | AUC=0.937601 | best=0.937601 | loss=0.28592 | 0.1 min ✓
Fold 2 | epoch 2/3 | AUC=0.938593 | best=0.938593 | loss=0.24578 | 0.1 min ✓
Fold 2 | epoch 3/3 | AUC=0.938863 | best=0.938863 | loss=0.23050 | 0.1 min ✓
Fold 2 DONE | AUC=0.938863273 | best_epoch=3 | total=0.4 min

RealMLP Mac | Fold 3/5 | device=mps
Fold 3 | epoch 1/3 | AUC=0.939116 | best=0.939116 | loss=0.28696 | 0.1 min ✓
Fold 3 | epoch 2/3 | AUC=0.939821 | best=0.939821 | loss=0.24629 | 0.1 min ✓
Fold 3 | epoch 3/3 | AUC=0.940066 | best=0.940066 | loss=0.23089 | 0.1 min ✓
Fold 3 DONE | AUC=0.940065642 | best_epoch=3 | total=0.4 min

RealMLP Mac 